In [4]:
import anndata as ad
import numpy as np
import pandas as pd

In [5]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

In [6]:
import pubchempy as pcp
from tqdm import tqdm

In [7]:
df_pert = pd.read_pickle('../../../lpm_style/lpm_style_embeddings_epoch_5/df_pert.pkl')

In [8]:
df_dili = pd.read_pickle('../../dili.pkl')

In [9]:
df_dili = df_dili.drop(columns=['cmap_name', 'symbol', 'code', 'symbol_', 'LPM_emb'])

In [10]:
df_dili['pubchem_cid_key'] = df_dili['pubchem_cid'].copy().astype(object)

In [11]:
df_dili.loc[~df_dili['pubchem_cid_key'].isna(), 'pubchem_cid_key'] = df_dili.loc[~df_dili['pubchem_cid_key'].isna(), 'pubchem_cid_key'].astype(int).astype(str)

In [12]:
df_dili

,pert_id,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,...,DILI_label_section,livertox_mechanism,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary,pubchem_cid,retrieved_smiles,ECFP:2,pubchem_cid_key
0,4-aminosalicylic acid,LT00505,4-aminosalicylic acid,Most-DILI-Concern,Warnings and precautions,5.0,NaN,Oral,DILIrank,NaN,...,Likely,NaN,NaN,NaN,NaN,NaN,4649.0,C1=CC(=C(C=C1N)O)C(=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4649
1,4-phenylbutyric acid,LT02343,4-phenylbutyric acid,Ambiguous DILI-concern,Adverse reactions,4.0,NaN,NaN,DILIrank,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4775.0,C1=CC=C(C=C1)CCCC(=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4775
2,Abacavir,LT00040,Abacavir,Most-DILI-Concern,Warnings and precautions,8.0,NaN,Oral,DILIrank,C,...,Likely,The cause of the clinically apparent hepatotox...,INTRODUCTION Abacavir sulfate is a nucleoside ...,2016 Jan 4,NaN,"Hepatitis, Cholestasis/Biliary, Hypersensitivi...",441300.0,C1CC1NC2=C3C(=NC(=N2)N)N(C=N3)C4CC(C=C4)CO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",441300
3,Abatacept,LT01402,Abatacept,Less-DILI-Concern,No match,0.0,NaN,Intravenous,DILIrank,C,...,Few cases,Abatacept is a recombinant human protein and a...,INTRODUCTION Abatacept is a recombinant fusion...,2021 Oct 6,idiosyncratic cases,"Hepatitis, Necrosis, Hypersensitivity, idiosyn...",NaN,NaN,None,NaN
4,Abciximab,LT01330,Abciximab,No-DILI-Concern,No match,0.0,NaN,Intravenous,DILIrank,NaN,...,No DILI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2467,Zinc,NaN,Zinc,NaN,NaN,NaN,NaN,NaN,NaN,E,...,No DILI,NaN,NaN,NaN,NaN,NaN,23994.0,[Zn],"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",23994
2468,Zolbetuximab,NaN,Zolbetuximab,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN
2469,Zoledronic acid,NaN,Zoledronic acid,NaN,NaN,NaN,NaN,NaN,NaN,C,...,Few cases,NaN,NaN,NaN,NaN,NaN,121586.0,C1=CN(C=N1)CC(O)(P(=O)(O)O)P(=O)(O)O.O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",121586
2470,Zoxazolamine,NaN,Zoxazolamine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,6103.0,C1=CC2=C(C=C1Cl)N=C(O2)N,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",6103


In [13]:
df_dili = df_dili.merge(df_pert, left_on='pubchem_cid_key', right_on='symbol', how='left')

In [14]:
df_dili = df_dili.drop(columns=['pubchem_cid_key'])

In [15]:
df_dili[~df_dili['lpm_style_embeddings'].isna()]

,pert_id,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,...,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary,pubchem_cid,retrieved_smiles,ECFP:2,symbol,code,lpm_style_embeddings
0,4-aminosalicylic acid,LT00505,4-aminosalicylic acid,Most-DILI-Concern,Warnings and precautions,5.0,NaN,Oral,DILIrank,NaN,...,NaN,NaN,NaN,NaN,4649.0,C1=CC(=C(C=C1N)O)C(=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4649,15925.0,"[0.013055342249572277, -0.03379404544830322, -..."
1,4-phenylbutyric acid,LT02343,4-phenylbutyric acid,Ambiguous DILI-concern,Adverse reactions,4.0,NaN,NaN,DILIrank,NaN,...,NaN,NaN,NaN,NaN,4775.0,C1=CC=C(C=C1)CCCC(=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4775,16305.0,"[0.44324803352355957, -0.09809990227222443, -0..."
2,Abacavir,LT00040,Abacavir,Most-DILI-Concern,Warnings and precautions,8.0,NaN,Oral,DILIrank,C,...,INTRODUCTION Abacavir sulfate is a nucleoside ...,2016 Jan 4,NaN,"Hepatitis, Cholestasis/Biliary, Hypersensitivi...",441300.0,C1CC1NC2=C3C(=NC(=N2)N)N(C=N3)C4CC(C=C4)CO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",441300,12052.0,"[0.008823661133646965, 0.029004085808992386, -..."
5,Acamprosate,LT01141,Acamprosate,Less-DILI-Concern,Adverse reactions,3.0,NaN,Oral,DILIrank,E,...,INTRODUCTION Acamprosate is a synthetic amino ...,2021 Sep 7,NaN,NaN,71158.0,CC(=O)NCCCS(=O)(=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",71158,33202.0,"[0.05246291309595108, 0.002570302691310644, -0..."
6,Acarbose,LT01034,Acarbose,Most-DILI-Concern,Warnings and precautions,8.0,NaN,Oral,DILIrank,B,...,INTRODUCTION Acarbose is an alpha glucosidase ...,2021 Jan 10,NaN,"Hepatitis, Cholestasis/Biliary, Immune-mediate...",41774.0,CC1C(C(C(C(O1)OC2C(OC(C(C2O)O)OC3C(OC(C(C3O)O)...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",41774,11563.0,"[0.038800906389951706, 0.01852630265057087, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2455,Vortioxetine,NaN,Vortioxetine,NaN,NaN,NaN,NaN,NaN,NaN,E*,...,INTRODUCTION Vortioxetine is a serotonergic an...,2020 Apr 8,NaN,NaN,9966051.0,CC1=CC(=C(C=C1)SC2=CC=CC=C2N3CCNCC3)C,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",9966051,37411.0,"[0.7780863642692566, 0.14110834896564484, 0.02..."
2457,Voxelotor,NaN,Voxelotor,NaN,NaN,NaN,NaN,NaN,NaN,E*,...,INTRODUCTION Voxelotor is an oral inhibitor of...,2021 Jul 12,NaN,NaN,71602803.0,CC(C)N1C(=CC=N1)C2=C(C=CC=N2)COC3=CC=CC(=C3C=O)O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",71602803,33461.0,"[0.010356931947171688, -0.008082606829702854, ..."
2463,Zanubrutinib,NaN,Zanubrutinib,NaN,NaN,NaN,NaN,NaN,NaN,E*,...,INTRODUCTION Zanubrutinib is an oral inhibitor...,2019 Dec 16,NaN,NaN,135565884.0,C=CC(=O)N1CCC(CC1)C2CCNC3=C(C(=NN23)C4=CC=C(C=...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",135565884,4140.0,"[-0.04659806564450264, 0.06169659644365311, -0..."
2469,Zoledronic acid,NaN,Zoledronic acid,NaN,NaN,NaN,NaN,NaN,NaN,C,...,NaN,NaN,NaN,NaN,121586.0,C1=CN(C=N1)CC(O)(P(=O)(O)O)P(=O)(O)O.O,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",121586,2550.0,"[0.012671688571572304, -0.08980398625135422, 0..."


In [16]:
df_dili.to_pickle("dili_lpm_style_embeddings_epoch5.pkl")